# Pipeline MLOps — Prédiction de Consommation Énergétique EDF

Ce notebook présente l'ensemble du pipeline **MLOps** pour la prédiction de la consommation électrique française,
en suivant les 5 DAGs déployés dans Airflow :

| # | DAG | Rôle |
|---|-----|------|
| 1 | **Data Ingestion** | Téléchargement, nettoyage, agrégation des données éCO2mix + Météo |
| 2 | **Training** | Feature engineering, entraînement multi-modèles, logging MLflow |
| 3 | **Batch Inference** | Chargement du modèle MLflow → prédiction batch → stockage PostgreSQL |
| 4 | **Benchmark** | Pipeline par région : features → modèle → prédictions régionalisées |
| 5 | **Performance Test** | Test de robustesse par injection de bruit gaussien progressif |

---
## Architecture

```mermaid
flowchart TD
    subgraph MLFlow
        G[MLFlow Registry]
    end
    subgraph Airflow
        subgraph Data Ingestion DAG
            T[API Open-Météo] -->|API Call| E[(weather_data)]
            A[Data éCO2mix] -->|Download| B[Fichiers Locaux]
            B -->|Ingestion| C[(eco2mix_raw)]
            C -->|Nettoyage| C1[(conso_clean)]
            C1 -->|Agrégation| D
            E -->|Agrégation| D[(aggregated_conso_weather)]
            D -->|Features Scaling| D1[(agg_conso_meteo_features)]
        end
        subgraph Training DAG
            D1 -->|Train/Evaluate| F[Modèles: RF, XGB, Prophet...]
            F -->|Register| G
        end
        subgraph Batch Inference DAG
            G -->|Load Model| H[Batch Prediction]
            D1 -->|Load Features| H
            H -->|Insert| I[(batch_predictions)]
        end
        subgraph Benchmark DAG
            D1 -->|Per Region| J[Features → Modèle → Prédictions]
            J -->|Regional| K[(batch_predictions_region)]
        end
        subgraph Performance Test DAG
            G -->|Load Model| L[Noise Injection Test]
            D1 --> L
            L -->|Log| G
        end
    end
```

---

## 0. Configuration & Imports

In [ ]:
# ------------------- Général -------------------
import requests, zipfile, io, os, re, unicodedata, time, json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ------------------- Data -------------------
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine

# ------------------- ML -------------------
import mlflow
import mlflow.sklearn
from mlflow.pyfunc import load_model
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb

# ------------------- Viz -------------------
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ------------------- Météo -------------------
import openmeteo_requests
import requests_cache
from retry_requests import retry

SEED = 42
np.random.seed(SEED)

In [ ]:
# ------------------- Chemins & Constantes -------------------
DATA_DIR = "../data"
DATA_DIR_PATH = Path(DATA_DIR)
BASE_URL = "https://eco2mix.rte-france.com/download/eco2mix/eCO2mix_RTE_Annuel-Definitif_{year}.zip"
SUPPORTED_EXTENSIONS = [".csv", ".xls", ".xlsx"]

START_YEAR = 2019
END_YEAR = 2022
TRAIN_YEARS = [2019, 2020, 2021]
TEST_YEARS = [2022]

# ------------------- DB Config -------------------
DB_CONFIG = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441,
}
# Connection SQLAlchemy pour pd.read_sql
DB_URI = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# ------------------- MLflow -------------------
MLFLOW_URL = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_URL)
MODEL_NAME = "MODEL_EDF"
TARGET = "consommation"

# ------------------- Columns -------------------
FEATURE_COLUMNS = ["temp_fr", "snow_fr", "hour", "month", "dayofweek", "weekend"]
UTILS_COLUMNS = ["id", "conso_id", "datetime", "year", "day"]

# Villes pour données météo
CITIES = {
    "Paris": (48.8566, 2.3522),
    "Marseille": (43.2965, 5.3698),
    "Bordeaux": (44.8378, -0.5792),
    "Brest": (48.3904, -4.4861),
    "Clermont-Ferrand": (45.7772, 3.0870),
    "Dijon": (47.3220, 5.0415),
    "Grenoble": (45.1885, 5.7245),
    "Lille": (50.6292, 3.0573),
    "Metz": (49.1193, 6.1757),
    "Montpellier": (43.6108, 3.8767),
    "Toulon": (43.1242, 5.9280),
    "Nancy": (48.6921, 6.1844),
    "Nantes": (47.2184, -1.5536),
    "Nice": (43.7102, 7.2620),
    "Orléans": (47.9029, 1.9093),
    "Rennes": (48.1173, -1.6778),
    "Rouen": (49.4431, 1.0993),
    "Saint-Etienne": (45.4397, 4.3872),
    "Strasbourg": (48.5734, 7.7521),
    "Toulouse": (43.6047, 1.4442),
    "Tours": (47.3941, 0.6848),
}

COLUMN_RENAME = {"hydraulique_fil_de_leeau_eclusee": "hydraulique_fil_de_leau_eclusee"}

EXPECTED_COLS = [
    "perimetre","nature","date","heures","consommation",
    "prevision_j_1","prevision_j","fioul","charbon","gaz",
    "nucleaire","eolien","solaire","hydraulique","pompage",
    "bioenergies","ech_physiques","taux_de_co2",
    "ech_comm_angleterre","ech_comm_espagne",
    "ech_comm_italie","ech_comm_suisse",
    "ech_comm_allemagne_belgique",
    "fioul_tac","fioul_cogen","fioul_autres",
    "gaz_tac","gaz_cogen","gaz_ccg","gaz_autres",
    "hydraulique_fil_de_leau_eclusee",
    "hydraulique_lacs","hydraulique_step_turbinage",
    "bioenergies_dechets","bioenergies_biomasse",
    "bioenergies_biogaz","stockage_batterie",
    "destockage_batterie","eolien_terrestre",
    "eolien_offshore","source_file"
]

---
## 1. DAG Ingestion — Data Ingestion Pipeline

Ce DAG télécharge les données historiques **éCO2mix** (RTE) et les données **météo** (Open-Meteo),
les nettoie, les agrège, et les stocke dans PostgreSQL.

**Flux :** `start → ingestion_job → cleanning_job → aggregation_job → complete`

### 1.1 Téléchargement des fichiers éCO2mix RTE

In [ ]:
def download_and_extract(start_year: int = 2012, target_dir: str = None) -> list[Path]:
    """
    Télécharge et dézippe les données éCO2mix annuelles depuis start_year.
    S'arrête dès qu'une année n'est plus disponible sur le serveur.
    """
    target_path = Path(target_dir) if target_dir else DATA_DIR_PATH
    target_path.mkdir(parents=True, exist_ok=True)
    extracted = []
    year = start_year

    while True:
        url = BASE_URL.format(year=year)
        print(f"Vérification : {url}")
        resp = requests.get(url)
        if resp.status_code != 200:
            print(f"Fin de téléchargement à l'année {year - 1}")
            break
        print(f"Téléchargement : {year}")
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            for member in zf.namelist():
                out = target_path / member
                if out.exists():
                    print(f"  Déjà présent : {out.name}")
                    continue
                zf.extract(member, target_path)
                extracted.append(out)
        year += 1
    return extracted


files = download_and_extract(start_year=START_YEAR, target_dir=str(DATA_DIR_PATH))
print(f"\nFichiers extraits : {len(files)}")

### 1.2 Fonctions de chargement et nettoyage

In [ ]:
def _clean_colname(col: str) -> str:
    if not isinstance(col, str):
        col = str(col)
    col = col.strip().replace("\ufffd", "e").replace("?", "e")
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = re.sub(r"[^a-z0-9]+", "_", col.lower()).strip("_")
    return col


def _load_single_file(filepath: Path) -> pd.DataFrame:
    suffix = filepath.suffix.lower()
    # Les .xls sont en fait des CSV avec séparateur variable
    def read_csv_smart(path):
        best, best_cols = None, 0
        for sep in [";", ",", "\t"]:
            try:
                df = pd.read_csv(path, sep=sep, encoding="latin1", low_memory=False, dtype=str)
                if df.shape[1] > best_cols:
                    best_cols = df.shape[1]
                    best = df
            except Exception:
                continue
        if best is None or best_cols == 1:
            raise ValueError("Impossible de détecter le séparateur CSV")
        return best

    if suffix == ".xls":
        df = read_csv_smart(filepath)
    else:
        raise ValueError(f"Format non supporté : {suffix}")

    df.columns = [_clean_colname(c) for c in df.columns]
    df["source_file"] = filepath.name
    return df


def load_all_data(data_dir: str) -> pd.DataFrame:
    """Charge tous les fichiers supportés du dossier data/."""
    data_path = Path(data_dir)
    if not data_path.exists():
        raise FileNotFoundError(f"Dossier introuvable : {data_dir}")
    all_dfs = []
    for f in data_path.iterdir():
        if f.suffix.lower() in SUPPORTED_EXTENSIONS:
            print(f"Chargement : {f.name}")
            all_dfs.append(_load_single_file(f))
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

### 1.3 Nettoyage et agrégation horaire

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().replace(["ND", "None", "nan", ""], pd.NA)
    df = df.rename(columns=COLUMN_RENAME).reindex(columns=EXPECTED_COLS)
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() > 0.3:
            df[col] = converted
        else:
            df[col] = df[col].astype("string").where(df[col].notna(), None)
    return df


def aggregate_hourly(df: pd.DataFrame) -> pd.DataFrame:
    """Agrège les données 15-min -> horaire (moyenne)."""
    df = df.copy().replace("ND", pd.NA)
    df["datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["heures"].astype(str), errors="coerce")
    df["hour_ts"] = df["datetime"].dt.floor("h")
    num = df.select_dtypes(include="number").columns
    non_num = df.columns.difference(num).difference(["datetime", "hour_ts"])
    agg = {c: "mean" for c in num} | {c: "first" for c in non_num}
    return df.groupby("hour_ts").agg(agg).reset_index().rename(columns={"hour_ts": "datetime"})

### 1.4 Ingestion des données météo (Open-Meteo Archive API)

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"


def fetch_weather(start_date: str, end_date: str) -> pd.DataFrame:
    """Récupère les données météo horaires pour toutes les villes."""
    all_cities = []
    for city, (lat, lon) in CITIES.items():
        params = {
            "latitude": lat, "longitude": lon,
            "start_date": start_date, "end_date": end_date,
            "hourly": ["temperature_2m", "relative_humidity_2m", "snowfall", "precipitation", "weather_code"],
            "timezone": "Europe/London",
        }
        response = openmeteo.weather_api(WEATHER_URL, params=params)[0]
        hourly = response.Hourly()
        data = {"date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()), inclusive="left"
        )}
        data["city"] = city
        data["temperature_2m"] = hourly.Variables(0).ValuesAsNumpy()
        data["relative_humidity_2m"] = hourly.Variables(1).ValuesAsNumpy()
        data["snowfall"] = hourly.Variables(2).ValuesAsNumpy()
        data["precipitation"] = hourly.Variables(3).ValuesAsNumpy()
        data["weather_code"] = hourly.Variables(4).ValuesAsNumpy()
        all_cities.append(pd.DataFrame(data))
        print(f"  Météo récupérée : {city}")
    return pd.concat(all_cities, ignore_index=True)


# Exemple : récupération météo 2019-2022
# df_meteo = fetch_weather("2019-01-01", "2022-12-31")
# df_meteo.head()

### 1.5 Agrégation conso + météo → table finale

In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Crée les features temporelles et nettoie."""
    df = aggregate_hourly(df)
    df["year"] = df["datetime"].dt.year
    df["hour"] = df["datetime"].dt.hour
    df["day"] = df["datetime"].dt.day
    df["month"] = df["datetime"].dt.month
    df["dayofweek"] = df["datetime"].dt.dayofweek
    df["weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    df.drop(columns=["perimetre", "nature", "source_file", "date", "heures", "datetime"],
            errors="ignore", inplace=True)
    return df.fillna(0)


# --- Pipeline ingestion complet (hors DB) ---
df_conso_raw = load_all_data(DATA_DIR)
print(f"\nShape brute : {df_conso_raw.shape}")

df_conso_clean = clean_dataframe(df_conso_raw)
df_conso_feat = create_features(df_conso_clean)
print(f"Shape features : {df_conso_feat.shape}")
df_conso_feat.head(3)

---
## 2. DAG Training — Pipeline d'Entraînement

Ce DAG charge les données agrégées depuis PostgreSQL, applique la sélection de features,
entraîne 5 modèles (Régression Linéaire, Random Forest, KNN, XGBoost, Prophet),
évalue et enregistre le meilleur dans **MLflow**.

**Flux :** `start → prepare_data_job → train_model_task → complete`

### 2.1 Chargement des features depuis PostgreSQL

In [ ]:
def load_data_from_db(source_table="aggregated_conso_weather",
                      selected_years=None, selected_months=None, selected_days=None):
    """Charge les données depuis PostgreSQL avec filtres année/mois/jour."""
    engine = create_engine(DB_URI)
    query = f"SELECT * FROM {source_table}"
    conditions = []
    if selected_years:
        conditions.append(f"year IN ({','.join(map(str, selected_years))})")
    if selected_months:
        conditions.append(f"month IN ({','.join(map(str, selected_months))})")
    if selected_days:
        conditions.append(f"day IN ({','.join(map(str, selected_days))})")
    if conditions:
        query += " WHERE " + " AND ".join(conditions)
    query += " ORDER BY year, month, day, hour"
    print(f"Exécution : {query}")
    return pd.read_sql(query, engine)


# Simulation avec les données locales si la DB n'est pas accessible
# df = load_data_from_db(selected_years=[2020, 2021, 2022])
# df.head()

### 2.2 Préparation des données d'entraînement

In [ ]:
def prepare_training_data(df, feature_columns=None, target=TARGET):
    """Extrait X (features) et y (cible) depuis le DataFrame chargé."""
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS
    df[target] = pd.to_numeric(df[target], errors="coerce")
    X = df[feature_columns].fillna(0)
    y = df[target].fillna(0)
    return X, y


### 2.3 Entraînement multi-modèles

In [ ]:
def train_models(X, y):
    """Entraîne 5 modèles et retourne un dictionnaire {nom: modèle}."""
    models = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        "KNN": KNeighborsRegressor(n_neighbors=7),
        "XGBoost": xgb.XGBRegressor(random_state=42, verbosity=0),
    }
    trained = {}
    for name, model in models.items():
        print(f"Entraînement : {name}...")
        model.fit(X, y)
        trained[name] = model
    return trained


def evaluate_model(model, X_test, y_test):
    """Calcule R², RMSE, MAPE."""
    y_pred = model.predict(X_test)
    return {
        "R2": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAPE (%)": mean_absolute_percentage_error(y_test, y_pred) * 100,
    }

### 2.4 Pipeline d'entraînement complet + Logging MLflow

In [ ]:
def run_training_pipeline(df, feature_columns=None, experiment_name="EDF_Training"):
    """Pipeline complet : préparation → entraînement → évaluation → MLflow."""
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS

    X, y = prepare_training_data(df, feature_columns)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Entraînement sur {len(X_train)} échantillons, test sur {len(X_test)}")
    print(f"Features : {feature_columns}\n")

    models = train_models(X_train, y_train)

    # Évaluation et classement
    rankings = []
    for name, model in models.items():
        metrics = evaluate_model(model, X_test, y_test)
        rankings.append((metrics["R2"], name, model))
        print(f"  {name:<20} R²={metrics['R2']:.4f}  RMSE={metrics['RMSE']:.2f}  MAPE={metrics['MAPE (%)']:.2f}%")

    rankings.sort(key=lambda x: x[0], reverse=True)
    best_r2, best_name, best_model = rankings[0]

    print(f"\n🏆 Champion : {best_name} (R²={best_r2:.4f})")

    # --- MLflow ---
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run() as run:
        mlflow.log_param("model_type", best_name)
        mlflow.log_param("n_features", len(feature_columns))
        mlflow.log_param("features", str(feature_columns))
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_metric("R2", best_r2)
        mlflow.sklearn.log_model(best_model, artifact_path=MODEL_NAME)
        model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
        mlflow.register_model(model_uri, MODEL_NAME)
        print(f"Modèle enregistré dans MLflow : {MODEL_NAME} (run={run.info.run_id})")

    return best_name, best_model, rankings


# --- Exécution avec les données locales ---
print("=" * 60)
print("LANCEMENT DU PIPELINE D'ENTRAÎNEMENT")
print("=" * 60)
best_name, best_model, rankings = run_training_pipeline(df_conso_feat)

---
## 3. DAG Batch Inference — Prédiction Batch

Ce DAG charge le meilleur modèle depuis **MLflow**, exécute des prédictions sur les données
préparées, et stocke les résultats dans PostgreSQL.

**Flux :** `start → prepare_data_job → batch_prediction_job → complete`

In [ ]:
def run_batch_prediction(df_source, feature_columns=None, model_name=MODEL_NAME,
                         mlflow_url=MLFLOW_URL):
    """
    Charge le modèle depuis MLflow, prédit sur df_source, retourne les prédictions.
    """
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS

    # Chargement du modèle depuis MLflow
    mlflow.set_tracking_uri(mlflow_url)
    client = MlflowClient()
    versions = client.get_latest_versions(model_name, stages=["None", "Staging", "Production"])
    if not versions:
        raise ValueError(f"Aucune version trouvée pour le modèle {model_name}")
    latest = max(int(v.version) for v in versions)
    model_uri = f"models:/{model_name}/{latest}"
    print(f"Chargement du modèle {model_name} v{latest} depuis {model_uri}")
    model = load_model(model_uri)

    # Préparation des features
    X = df_source[feature_columns].copy()
    X = X.replace("ND", pd.NA).fillna(0)
    print(f"Prédiction sur {len(X)} échantillons\n")

    predictions = model.predict(X)

    # Assemblage du résultat
    result = df_source[["datetime"] + feature_columns].copy() if "datetime" in df_source.columns else df_source.copy()
    result["prediction"] = predictions
    result["model_version"] = latest

    print("Aperçu des prédictions :")
    display(result.head())

    return result


# --- Exemple avec les features locales ---
print("=" * 60)
print("BATCH INFERENCE")
print("=" * 60)

# Simuler le chargement depuis la DB
# df_features = load_data_from_db(source_table="agg_conso_meteo_features", selected_years=[2022])

# Fallback : utiliser les données en mémoire
df_inference = df_conso_feat[df_conso_feat["year"].isin([2022])].copy()
print(f"Données pour inférence : {len(df_inference)} lignes (année 2022)\n")

try:
    df_pred = run_batch_prediction(df_inference)
except Exception as e:
    print(f"Erreur (normal si MLflow n'est pas actif) : {e}")

---
## 4. DAG Benchmark — Pipeline par Région

Ce DAG décline le pipeline complet **par région** : construction des tables de features
régionalisées, entraînement de modèles spécifiques, et prédictions régionalisées.

**Flux :** `start → build_feature_tables → train_region_models → predict_consumption → complete`

In [ ]:
REGIONS = ["Paris", "Marseille", "Bordeaux", "Brest", "Lille", "Lyon",
           "Nancy", "Nantes", "Nice", "Rennes", "Strasbourg", "Toulouse"]


def prepare_region_features(df_weather, region: str, feature_columns=None):
    """Filtre les données météo pour une région spécifique."""
    if feature_columns is None:
        feature_columns = ["temp_fr", "hum_fr", "snow_fr", "rain_fr"]

    # Filtrer par ville
    df_region = df_weather[df_weather["city"] == region].copy()
    df_region["datetime"] = pd.to_datetime(df_region["datetime"])

    # Features temporelles
    df_region["hour"] = df_region["datetime"].dt.hour
    df_region["month"] = df_region["datetime"].dt.month
    df_region["dayofweek"] = df_region["datetime"].dt.dayofweek
    df_region["weekend"] = df_region["dayofweek"].isin([5, 6]).astype(int)

    return df_region[["datetime"] + [c for c in feature_columns if c in df_region.columns]].fillna(0)


def benchmark_region_pipeline(region: str, df_weather_region, df_conso):
    """Pipeline benchmark complet pour une région."""
    print(f"\n{'=' * 50}")
    print(f"Région : {region}")
    print(f"=" * 50)

    # Fusion conso + météo région
    df_merged = df_conso.merge(df_weather_region, on="datetime", how="left").fillna(0)

    # Features disponibles
    available_features = [c for c in FEATURE_COLUMNS if c in df_merged.columns]
    print(f"Features : {available_features}")

    # Entraînement
    X, y = prepare_training_data(df_merged, available_features)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = train_models(X_train, y_train)

    # Évaluation
    results = []
    for name, model in models.items():
        metrics = evaluate_model(model, X_test, y_test)
        results.append((metrics["R2"], name, metrics))
        print(f"  {name:<20} R²={metrics['R2']:.4f}  RMSE={metrics['RMSE']:.2f}")

    results.sort(key=lambda x: x[0], reverse=True)
    print(f"  → Meilleur : {results[0][1]} (R²={results[0][0]:.4f})")
    return {"region": region, "best_model": results[0][1], "best_r2": results[0][0]}


# --- Exemple : benchmark sur quelques régions ---
# Note : nécessite les données météo brutes (par ville)
print("Benchmark par région — aperçu du pipeline")
print("(nécessite les données météo brutes par ville pour exécution complète)\n")

# benchmark_region_pipeline("Paris", df_meteo_paris, df_conso_feat)

---
## 5. DAG Performance Test — Test de Robustesse

Ce DAG évalue la robustesse du modèle en injectant du **bruit gaussien progressif**
sur les features et en mesurant la dégradation des métriques.

**Flux :** `start → check_data_availability → run_performance_test → complete`

In [ ]:
def add_noise_to_features(X: pd.DataFrame, noise_level: float, seed: int = 42) -> pd.DataFrame:
    """Ajoute un bruit gaussien aux features numériques."""
    np.random.seed(seed)
    X_noisy = X.copy()
    for col in X_noisy.select_dtypes(include=[np.number]).columns:
        col_std = X_noisy[col].std()
        if col_std > 0:
            noise = np.random.normal(0, noise_level * col_std, size=X_noisy[col].shape)
            X_noisy[col] = X_noisy[col] + noise
    return X_noisy


def run_performance_test(model, X_test, y_test,
                         noise_levels=None):
    """
    Teste le modèle avec différents niveaux de bruit.
    Retourne les métriques par niveau + dégradation.
    """
    if noise_levels is None:
        noise_levels = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    results = {"noise_levels": noise_levels, "metrics": {"R2": [], "RMSE": [], "MAPE (%)": []},
               "degradation": {"R2": [], "RMSE": [], "MAPE (%)": []}, "baseline": None}

    for i, noise in enumerate(noise_levels):
        X_noisy = add_noise_to_features(X_test, noise, seed=i) if noise > 0 else X_test.copy()
        metrics = evaluate_model(model, X_noisy, y_test)
        results["metrics"]["R2"].append(metrics["R2"])
        results["metrics"]["RMSE"].append(metrics["RMSE"])
        results["metrics"]["MAPE (%)"].append(metrics["MAPE (%)"])

        if i == 0:
            results["baseline"] = metrics.copy()

    # Calcul de la dégradation
    base = results["baseline"]
    for i in range(len(noise_levels)):
        results["degradation"]["R2"].append(
            0 if i == 0 else ((results["metrics"]["R2"][i] - base["R2"]) / abs(base["R2"])) * 100)
        results["degradation"]["RMSE"].append(
            0 if i == 0 else ((results["metrics"]["RMSE"][i] - base["RMSE"]) / base["RMSE"]) * 100)
        results["degradation"]["MAPE (%)"].append(
            0 if i == 0 else ((results["metrics"]["MAPE (%)"][i] - base["MAPE (%)"]) / base["MAPE (%)"]) * 100)

    return results


def plot_performance_results(results):
    """Affiche les graphiques de performance et dégradation."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Graphique 1 : Métriques brutes
    ax = axes[0]
    ax.plot(results["noise_levels"], results["metrics"]["R2"], "o-", label="R²", linewidth=2)
    ax.plot(results["noise_levels"], results["metrics"]["RMSE"], "s-", label="RMSE", linewidth=2)
    ax.set_xlabel("Niveau de bruit")
    ax.set_ylabel("Score")
    ax.set_title("Métriques brutes vs Bruit")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Graphique 2 : Dégradation relative
    ax = axes[1]
    ax.plot(results["noise_levels"], results["degradation"]["R2"], "o-", label="R² dégradation", linewidth=2)
    ax.plot(results["noise_levels"], results["degradation"]["RMSE"], "s-", label="RMSE dégradation", linewidth=2)
    ax.plot(results["noise_levels"], results["degradation"]["MAPE (%)"], "^-", label="MAPE dégradation", linewidth=2)
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax.axhline(y=-10, color="orange", linestyle=":", alpha=0.7, label="Seuil avertissement (-10%)")
    ax.axhline(y=-20, color="red", linestyle=":", alpha=0.7, label="Seuil critique (-20%)")
    ax.set_xlabel("Niveau de bruit")
    ax.set_ylabel("Dégradation relative (%)")
    ax.set_title("Dégradation des métriques vs Bruit")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def display_performance_table(results):
    """Affiche un tableau récapitulatif des performances."""
    df = pd.DataFrame({
        "Noise": results["noise_levels"],
        "R²": [f"{v:.4f}" for v in results["metrics"]["R2"]],
        "RMSE": [f"{v:.2f}" for v in results["metrics"]["RMSE"]],
        "MAPE (%)": [f"{v:.2f}" for v in results["metrics"]["MAPE (%)"]],
        "R² Deg (%)": [f"{v:.2f}" for v in results["degradation"]["R2"]],
        "RMSE Deg (%)": [f"{v:.2f}" for v in results["degradation"]["RMSE"]],
        "MAPE Deg (%)": [f"{v:.2f}" for v in results["degradation"]["MAPE (%)"]],
    })
    display(df)
    return df

### 5.1 Exécution du test de performance

In [ ]:
print("=" * 60)
print("TEST DE PERFORMANCE — INJECTION DE BRUIT")
print("=" * 60)

# Utiliser le meilleur modèle entraîné précédemment
if 'best_model' in globals():
    # Préparer les données de test
    df_test = df_conso_feat[df_conso_feat["year"] == 2022].copy()
    X_test, y_test = prepare_training_data(df_test)
    X_test = X_test.head(500)
    y_test = y_test.head(500)

    print(f"\nTest sur {len(X_test)} échantillons (année 2022)")
    print(f"Modèle utilisé : {best_name}\n")

    # Exécution du test
    perf_results = run_performance_test(best_model, X_test, y_test)

    # Résultats
    print("\nTableau des performances :")
    df_perf = display_performance_table(perf_results)

    # Graphiques
    print("\nGraphiques de performance :")
    plot_performance_results(perf_results)

else:
    print("Aucun modèle entraîné disponible. Exécutez d'abord la section Training.")

---
## Résumé du Pipeline MLOps

| Étape | DAG Airflow | Techno | Sortie |
|-------|-------------|--------|--------|
| **1. Ingestion** | `data_ingestion_dag` | Python, PostgreSQL, Open-Meteo API | Tables `aggregated_conso_weather`, `agg_conso_meteo_features` |
| **2. Training** | `training_dag` | scikit-learn, XGBoost, MLflow | Modèle enregistré dans MLflow Registry (`MODEL_EDF`) |
| **3. Batch Inference** | `batch_prediction_dag` | MLflow, PostgreSQL | Table `batch_predictions_*` avec prédictions |
| **4. Benchmark** | `batch_prediction_dag_by_region` | Par région, MLflow | Modèles et prédictions régionalisés |
| **5. Performance** | `performance_test_dag` | MLflow, bruit gaussien | Métriques de robustesse, graphiques, alertes |

### Infrastructure

- **Orchestration** : Apache Airflow (ports 8009)
- **Model Registry** : MLflow (port 5000)
- **Base de données** : PostgreSQL (port 5441)
- **Docker** : `docker-compose up -d` pour tout déployer